# Hotel Booking Demand: Cancellation Prediction

## Project Overview

**Objective:** Predict whether a hotel booking will be canceled (`is_canceled`) using booking-related information available for the prediction task.

**Workflow:** Data understanding → cleaning → feature engineering → focused EDA → preprocessing → model training → evaluation → feature importance → conclusion.

**Target:** `is_canceled` (`0` = not canceled, `1` = canceled).

> **Leakage consideration:** Reservation outcome fields such as `reservation_status` and `reservation_status_date` are removed because they reveal information about the booking outcome. Features that are not available at booking time are excluded from the model as well.

## 1. Imports and Data Loading
Load the dataset and inspect its basic structure before making changes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/hotel_bookings.csv')

In [ ]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.info()

In [ ]:
print("Missing values in each column:")
# 5. Missing values
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.describe()

In [ ]:
# 7. Target distribution
df['is_canceled'].value_counts()

## 2. Data Cleaning
We check exact duplicates, confirm that duplicate groups do not contain conflicting target labels, and remove exact duplicate rows. This avoids counting repeated records as independent examples.

In [ ]:
# Check whether identical feature records have conflicting target labels.
duplicate_rows = df[df.duplicated(keep=False)]
duplicate_target_check = (
    duplicate_rows
    .groupby(list(df.columns.drop("is_canceled"))) ["is_canceled"]
    .nunique()
)

print(
    "Duplicate groups with conflicting targets:",
    (duplicate_target_check > 1).sum()
)

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

In [ ]:
missing_pct = (df.isnull().mean() * 100)

missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing %": missing_pct
})

missing_summary = (
    missing_summary[missing_summary["Missing Values"] > 0]
    .sort_values("Missing %", ascending=False)
)

missing_summary

### Missing-value handling

- `children`: four missing values are filled with `0`.
- `company` and `agent`: missingness indicators are created because missing IDs may themselves carry information; the raw identifiers are excluded from modeling later.
- `country`: missing values are represented as an explicit `Unknown` category.

The raw `company` and `agent` columns remain temporarily for transparency but are removed from the feature matrix later.

In [ ]:
# Handle missing values in children
df["children"] = df["children"].fillna(0)

print("Missing values in children:", df["children"].isna().sum())

In [ ]:
# Create missingness indicators for ID-based features

df["company_missing"] = df["company"].isna().astype(int)
df["agent_missing"] = df["agent"].isna().astype(int)

# Treat missing country as an explicit category
df["country"] = df["country"].fillna("Unknown")

print("Missing company:", df["company"].isna().sum())
print("Missing agent:", df["agent"].isna().sum())
print("Missing country:", df["country"].isna().sum())

### Remove outcome leakage
The reservation status fields describe the eventual outcome and must not be used to predict that same outcome.

In [ ]:
# Remove features that contain information about the final booking outcome
df = df.drop(columns=["reservation_status", "reservation_status_date"])

print("Shape after removing leakage features:", df.shape)
print("\nRemaining columns:")
print(df.columns.tolist())

## 3. Feature Engineering
Create compact features that summarize stay length, party size, guest-nights, and arrival timing.

In [ ]:
# Create total length of stay
df["total_stay_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

print(df["total_stay_nights"].describe())

In [ ]:
# Create total number of guests
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

# Flag bookings with no recorded guests
df["zero_guests"] = (df["total_guests"] == 0).astype(int)

print("Total guests summary:")
print(df["total_guests"].describe())

print("\nZero-guest bookings:", df["zero_guests"].sum())

In [ ]:
# Create total guest-nights
df["total_guest_nights"] = (
    df["total_guests"] * df["total_stay_nights"]
)

print(df["total_guest_nights"].describe())

In [ ]:
# Create a proper arrival date
df["arrival_date"] = pd.to_datetime(
    df["arrival_date_year"].astype(str)
    + "-"
    + df["arrival_date_month"]
    + "-"
    + df["arrival_date_day_of_month"].astype(str),
    format="%Y-%B-%d"
)

print(df["arrival_date"].head())
print("\nDate range:")
print(df["arrival_date"].min(), "to", df["arrival_date"].max())

In [ ]:
# Extract day of week
df["arrival_day_of_week"] = df["arrival_date"].dt.dayofweek

# Monday = 0 ... Sunday = 6
df["arrival_weekend"] = (
    df["arrival_day_of_week"] >= 5
).astype(int)

print(
    df[
        ["arrival_date", "arrival_day_of_week", "arrival_weekend"]
    ].head(10)
)

In [ ]:
# Remove redundant date representations
df.drop(
    columns=[
        "arrival_date",
        "arrival_date_day_of_month",
        "arrival_weekend"
    ],
    inplace=True
)

print("Shape:", df.shape)

print("\nDate-related columns:")
print([
    col for col in df.columns
    if "arrival" in col
])

### Rare-country grouping

Countries with fewer than 20 bookings are grouped into `Other` to reduce the number of very-low-frequency categories. **Methodological note:** in a stricter production workflow, the category frequency mapping should be learned from training data only and then applied to validation/test data. Here, this existing exploratory workflow groups categories before the split, so this is a small preprocessing leakage limitation to disclose or correct in a future refinement.

In [ ]:
country_threshold = 20

country_counts = df["country"].value_counts()

df["country"] = df["country"].where(
    df["country"].map(country_counts) >= country_threshold,
    "Other"
)

print("Unique country categories after grouping:", df["country"].nunique())

print("\nTop countries:")
print(df["country"].value_counts().head(20))

print("\nMissing values:", df["country"].isna().sum())

## 4. Focused Exploratory Data Analysis
The following checks focus on class distribution and selected booking characteristics relevant to cancellation prediction. EDA findings describe associations, not causation.

In [ ]:
import matplotlib.pyplot as plt

target_counts = df["is_canceled"].value_counts()

print("Target distribution:")
print(target_counts)

print("\nTarget percentage:")
print(
    (df["is_canceled"].value_counts(normalize=True) * 100).round(2)
)

plt.figure(figsize=(7, 5))

target_counts.plot(kind="bar")

plt.title("Booking Cancellation Distribution")
plt.xlabel("Cancellation Status")
plt.ylabel("Number of Bookings")
plt.xticks(
    ticks=[0, 1],
    labels=["Not Canceled", "Canceled"],
    rotation=0
)

plt.show()

### Numerical feature overview

In [ ]:
numerical_features = [
    "lead_time",
    "total_stay_nights",
    "total_guests",
    "total_guest_nights",
    "adr",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "required_car_parking_spaces",
    "total_of_special_requests"
]

df[numerical_features].describe().T

### Cancellation rate by lead-time group
Longer lead times showed higher cancellation rates in this dataset. This is an association, not evidence that lead time itself causes cancellation.

In [ ]:
df["lead_time_group"] = pd.cut(
    df["lead_time"],
    bins=[-1, 7, 30, 90, 180, 365, float("inf")],
    labels=[
        "0-7 days",
        "8-30 days",
        "31-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

lead_time_cancel = (
    df.groupby("lead_time_group", observed=True)["is_canceled"]
    .mean()
    .mul(100)
    .round(2)
)

print("Cancellation rate by lead time:")
print(lead_time_cancel)

### Cancellation patterns across booking characteristics

In [ ]:
hotel_cancel = (
    df.groupby("hotel")["is_canceled"]
    .mean()
    .mul(100)
    .round(2)
)

print("Cancellation rate by hotel:")
print(hotel_cancel)

In [ ]:
deposit_cancel = (
    df.groupby("deposit_type")["is_canceled"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("Cancellation rate by deposit type:")
print(deposit_cancel)

In [ ]:
segment_cancel = (
    df.groupby("market_segment")["is_canceled"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("Cancellation rate by market segment:")
print(segment_cancel)

In [ ]:
customer_cancel = (
    df.groupby("customer_type")["is_canceled"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("Cancellation rate by customer type:")
print(customer_cancel)

In [ ]:
month_cancel = (
    df.groupby("arrival_date_month")["is_canceled"]
    .mean()
    .mul(100)
    .round(2)
)

month_order = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

month_cancel = month_cancel.reindex(month_order)

print("Cancellation rate by arrival month:")
print(month_cancel)

## 5. Feature Selection
Define the target and remove fields that are unavailable at booking time, identifiers that are not retained as raw features, and the target itself from `X`.

In [ ]:
# Features to exclude from the booking-time prediction model

drop_features = [
    "is_canceled",
    "assigned_room_type",
    "booking_changes",
    "days_in_waiting_list",
    "company",
    "agent"
]

X = df.drop(columns=drop_features)
y = df["is_canceled"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nRemaining features:")
print(X.columns.tolist())

## 6. Train-Test Split
Use a stratified 80:20 split so the target-class proportions remain similar in the training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTest set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTarget distribution:")
print("Train:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest:")
print(y_test.value_counts(normalize=True).round(4))

## 7. Preprocessing
Numerical features are standardized and categorical features are one-hot encoded. `handle_unknown="ignore"` allows the encoder to handle categories that appear in the test set but were not observed during fitting.

In [ ]:
categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumber of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of numerical features:", len(numerical_features))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),  #z=x−μ/​σ
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully.")

## 8. Model 1 — Logistic Regression
Logistic Regression provides a baseline classifier. The preprocessing is included inside the pipeline so it is fitted on training data when the pipeline is trained.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Predictions
y_pred = logistic_pipeline.predict(X_test)

# Probabilities for ROC-AUC
y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression Performance")
print("-" * 40)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 9. Model 2 — Random Forest
Random Forest can learn nonlinear relationships and interactions. It uses the same preprocessing and train-test split as Logistic Regression for a more consistent comparison.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_pipeline.fit(X_train, y_train)

print("Random Forest trained successfully.")

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Predictions using the default threshold of 0.5
y_pred_rf = rf_pipeline.predict(X_test)

# Probability estimates for the cancellation class
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest Performance")
print("-" * 40)

print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_rf):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

## 10. Model Comparison
Compare both models on the same held-out test set using accuracy, precision, recall, F1-score, and ROC-AUC. Classification metrics use the default probability threshold of 0.50.

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Predictions from both trained pipelines
y_pred_lr = logistic_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

# Predicted probabilities for the cancellation class
y_prob_lr = logistic_pipeline.predict_proba(X_test)[:, 1]
y_prob_rf = rf_pipeline.predict_proba(X_test)[:, 1]

# Compare models at the default threshold of 0.50
results = []

models = {
    "Logistic Regression": (y_pred_lr, y_prob_lr),
    "Random Forest": (y_pred_rf, y_prob_rf)
}

for model_name, (y_pred, y_prob) in models.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index("Model")

print("Final Model Comparison — Default Threshold (0.50)")
display(results_df.round(4))

## 11. Feature Importance
The Random Forest importance scores below are impurity-based importances. They indicate how the fitted model used features for tree splits; they do not establish causation or show the direction of a relationship. One-hot encoded categories appear as separate features.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Retrieve the fitted Random Forest model
rf_model = rf_pipeline.named_steps["model"]

# Retrieve feature names after preprocessing
feature_names = (
    rf_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

# Extract feature importance
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
})

# Show the 15 most important transformed features
top_features = (
    importance_df
    .sort_values("importance", ascending=False)
    .head(15)
)

print("Top 15 Features:")
print(top_features.to_string(index=False))

# Visualize the results
plt.figure(figsize=(10, 7))

plt.barh(
    top_features["feature"].iloc[::-1],
    top_features["importance"].iloc[::-1]
)

plt.title("Random Forest — Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 12. Final Conclusion

### Selected model: Random Forest Classifier

The Random Forest achieved stronger results than Logistic Regression on the held-out test set across the reported metrics:

| Metric | Logistic Regression | Random Forest |
|---|---:|---:|
| Accuracy | 79.05% | 82.50% |
| Precision | 67.33% | 75.65% |
| Recall | 46.20% | 53.59% |
| F1-score | 54.80% | 62.74% |
| ROC-AUC | 83.39% | 88.76% |

Random Forest is selected as the leading model for this experiment. Its recall at the default threshold is 53.59%, meaning it misses a substantial share of actual cancellations. Threshold choice should depend on the relative business cost of missed cancellations and false alarms.

### Limitations

- The evaluation is based on one stratified train-test split; performance on future bookings may differ.
- Rare-country grouping was performed before the split, so category grouping has seen the overall feature distribution. A stricter workflow would learn this mapping from training data only.
- A threshold of 0.35 was explored using the test set in the original exploratory notebook. Those threshold-specific results are exploratory and are not an unbiased final test estimate.
- Impurity-based feature importance is model-specific and should not be interpreted as causal influence.
- Before deployment, validate on temporally newer bookings and choose a decision threshold using validation data and business cost considerations.